In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
HERE = Path.cwd()
DATA_ROOT = HERE.parent
ACTUAL_DIR = DATA_ROOT / "actual"
OUT_DIR = HERE

LEAGUES = ["bundesliga", "la_liga", "premier_league", "serie_a"]

In [3]:
def paths_for(league: str):
    matches_all_seeds = OUT_DIR / f"{league}_simulated_matches_all_seeds.csv"
    actual_standings = ACTUAL_DIR / f"{league}_standings_all_seasons.csv"
    out_csv = OUT_DIR / f"{league}_simulated_standings_all_seasons.csv"
    return matches_all_seeds, actual_standings, out_csv

In [4]:
def load_actual_standings(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    if "team_name" in df.columns:
        df = df.rename(columns={"team_name": "team"})

    df = df.rename(columns={"rank": "actual_rank"})
    df = df[["season", "team", "actual_rank"]].sort_values(
        ["season", "actual_rank", "team"]
    ).reset_index(drop=True)
    return df

In [5]:
def ranks_from_matches_all_seeds(matches_csv: Path) -> pd.DataFrame:
    df = pd.read_csv(matches_csv)
    seed_cols = [c for c in df.columns if c.startswith("simulated_home_team_result_seed_")]

    def seed_points_to_ranks(col: str) -> pd.DataFrame:
        vals = df[col].values

        # points: home team
        home_pts = np.where(vals == 1, 3, np.where(vals == 0, 1, 0))
        # points: away team
        away_pts = np.where(vals == -1, 3, np.where(vals == 0, 1, 0))

        home_tbl = pd.DataFrame({"season": df["season"], "team": df["home_team"], "points": home_pts})
        away_tbl = pd.DataFrame({"season": df["season"], "team": df["away_team"], "points": away_pts})
        pts = pd.concat([home_tbl, away_tbl], ignore_index=True)
        pts = pts.groupby(["season", "team"], as_index=False)["points"].sum()

        # rank per season
        pts = pts.sort_values(["season", "points", "team"], ascending=[True, False, True])
        pts[f"simulated_rank_{col.split('_seed_')[-1]}"] = pts.groupby("season").cumcount() + 1
        return pts[["season", "team", f"simulated_rank_{col.split('_seed_')[-1]}"]]
    
    ranks = None
    for c in seed_cols:
        r = seed_points_to_ranks(c)
        ranks = r if ranks is None else ranks.merge(r, on=["season", "team"], how="outer")

    return ranks.sort_values(["season", "team"]).reset_index(drop=True)

In [6]:
summary = []
for lg in LEAGUES:
    matches_csv = OUT_DIR / f"{lg}_simulated_matches_all_seeds.csv"
    actual_csv  = ACTUAL_DIR / f"{lg}_standings_all_seasons.csv"
    out_csv     = OUT_DIR / f"{lg}_simulated_standings_all_seasons.csv"

    actual = load_actual_standings(actual_csv)
    sim_ranks = ranks_from_matches_all_seeds(matches_csv)

    combined = actual.merge(sim_ranks, on=["season", "team"], how="left")
    combined.to_csv(out_csv, index=False)

    summary.append({"league": lg, "rows": len(combined), "output": out_csv.name})

pd.DataFrame(summary)

,league,rows,output
0,bundesliga,378,bundesliga_simulated_standings_all_seasons.csv
1,la_liga,460,la_liga_simulated_standings_all_seasons.csv
2,premier_league,440,premier_league_simulated_standings_all_seasons...
3,serie_a,454,serie_a_simulated_standings_all_seasons.csv
